# Socio-Technical Music Fan Persona Clustering

This notebook segments music fans based on their consumption behavior, platform preferences, and subgenre affinities. By identifying distinct listener personas, we can better understand how different groups experience algorithmic flattening and local content curation.

## 1. Environment Setup & Imports

We load libraries for data manipulation (`pandas`, `numpy`), dimensionality reduction (`prince`), clustering (`sklearn`), and path resolution.

In [ ]:
import json
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from prince import MCA
from sklearn.cluster import KMeans

## 2. Robust Path Loading & Ingestion

Instead of using hardcoded relative paths, we dynamically resolve the repository root to locate the processed dataset using the project's standardized path manager.

In [ ]:
# Resolve repository root and register src path
notebook_dir = Path(__file__).resolve().parent if "__file__" in locals() else Path.cwd()
repo_root = notebook_dir.parent
if str(repo_root / "src") not in sys.path:
    sys.path.append(str(repo_root / "src"))

from kulture.common.paths import get_dataset_path

# Load cleaned survey data
data_path = get_dataset_path()
df = pd.read_csv(data_path)
print(f"Loaded {len(df)} rows from {data_path.name}")

## 3. Preprocessing & Categorical Feature Simplification

Because primary platforms and top genres are multi-select inputs, they have high categorical cardinality. We one-hot encode binary indicators for key platforms and local/global subgenres to make them suitable for Multiple Correspondence Analysis (MCA).

In [ ]:
# Select and clean baseline categorical features
features = ['age_band', 'hours_streaming_weekly', 'discovery_method', 'primary_platforms', 'top_genres']
df_cluster = df[features].dropna()
df_cluster = df_cluster[(df_cluster != 'Unknown').all(axis=1)]
df_cluster = df_cluster[(df_cluster != 'No response').all(axis=1)]

# Binary indicators for platforms
df_cluster['has_spotify'] = df_cluster['primary_platforms'].str.contains('Spotify').astype(str)
df_cluster['has_apple'] = df_cluster['primary_platforms'].str.contains('Apple Music').astype(str)
df_cluster['has_youtube'] = df_cluster['primary_platforms'].str.contains('YouTube').astype(str)

# Binary indicators for genres
df_cluster['has_amapiano'] = df_cluster['top_genres'].str.contains('Amapiano').astype(str)
df_cluster['has_gospel'] = df_cluster['top_genres'].str.contains('Gospel').astype(str)
df_cluster['has_afrobeats'] = df_cluster['top_genres'].str.contains('Afrobeats').astype(str)
df_cluster['has_hiphop'] = df_cluster['top_genres'].str.contains('Hip-Hop/Rap').astype(str)
df_cluster['has_sahouse'] = df_cluster['top_genres'].str.contains('SA House').astype(str)

# Drop original high-cardinality multi-select columns
X = df_cluster.drop(columns=['primary_platforms', 'top_genres'])
X.head()

## 4. Multiple Correspondence Analysis (MCA)

MCA is the categorical equivalent of Principal Component Analysis (PCA). We project our high-dimensional categorical features onto a lower-dimensional continuous space to capture the principal dimensions of listener variance.

In [ ]:
mca = MCA(n_components=3, n_iter=10, random_state=42)
mca.fit(X)
coords = mca.transform(X)
print(f"Projected data shape: {coords.shape}")

## 5. K-Means Clustering on MCA Dimensions

We cluster the fans in the continuous MCA projection space into $K=3$ distinct cohorts using K-Means.

In [ ]:
n_clusters = 3
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
df_cluster['cluster'] = kmeans.fit_predict(coords)
print(df_cluster['cluster'].value_counts())

## 6. Profile Profiling & JSON Export

For each cluster, we identify the mode of each categorical feature to construct representative persona cards, then save the output to `persona_results.json`.

In [ ]:
results = {}
for i in range(n_clusters):
    cluster_data = df_cluster[df_cluster['cluster'] == i]
    size = len(cluster_data)
    
    summary = {}
    for col in X.columns:
        summary[col] = cluster_data[col].mode().iloc[0]
        
    results[f"Persona_{i+1}"] = {
        'size': size,
        'percent': float(size / len(df_cluster) * 100),
        'attributes': summary
    }

with open('persona_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print("Persona Clustering complete and saved.")